# Psychology RAFT-RAG — RAG Inference
This notebook retrieves trusted psychology knowledge with BGE-small and generates a grounded answer with Qwen.

In [ ]:
!pip -q install -U transformers==4.46.3 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1 sentence-transformers==3.1.1 pinecone==5.3.1 python-dotenv

In [ ]:
import os, torch
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
EMBED_MODEL = 'BAAI/bge-small-en-v1.5'
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER_PATH = '/content/psychology-qwen-qlora'
PINECONE_INDEX = os.environ.get('PINECONE_INDEX_NAME', 'psychology-knowledge')
TOP_K = 4
assert os.environ.get('PINECONE_API_KEY'), 'Set PINECONE_API_KEY in Colab Secrets/environment.'

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(PINECONE_INDEX)
print('Embedding dimension:', embedder.get_sentence_embedding_dimension())

## Safety layer
This is a basic demonstration guard, not a clinical risk classifier. High-risk messages should be routed to a human/crisis-support response instead of normal generation.

In [ ]:
HIGH_RISK_TERMS = ('suicide', 'kill myself', 'end my life', 'self harm', 'self-harm', 'want to die')
def safety_response(question):
    q = question.lower()
    if any(term in q for term in HIGH_RISK_TERMS):
        return ('I am sorry you are dealing with something this serious. I cannot provide emergency or clinical care. '
                'If you may act on these thoughts or are in immediate danger, contact your local emergency service now, '
                'go to the nearest emergency department, or contact a trusted person who can stay with you. A qualified mental-health professional can provide further support.')
    return None

In [ ]:
def retrieve(question, top_k=TOP_K):
    vector = embedder.encode(question, normalize_embeddings=True).tolist()
    result = index.query(vector=vector, top_k=top_k, include_metadata=True)
    return result.get('matches', [])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

In [ ]:
def answer(question):
    emergency = safety_response(question)
    if emergency:
        return emergency, []
    matches = retrieve(question)
    contexts=[]
    for m in matches:
        md=m.get('metadata') or {}
        text=md.get('text') or md.get('content') or ''
        if text: contexts.append(text)
    context='\\n\\n--- SOURCE ---\\n'.join(contexts)
    prompt=(f'You are a psychology psychoeducation assistant.\\nUse only the supplied context for factual claims. '
            f'Do not diagnose or claim to replace a professional. If the context is insufficient, say so.\\n\\nContext:\\n{context}\\n\\nQuestion: {question}')
    inputs=tokenizer.apply_chat_template([{'role':'user','content':prompt}], return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.no_grad(): output=model.generate(inputs, max_new_tokens=300, do_sample=True, temperature=0.3, top_p=0.9)
    text=tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    return text, matches

In [ ]:
question='What is stress, and what are some general ways to manage it?'
response, matches = answer(question)
print(response)
print('\\nRetrieved sources:', len(matches))
for i,m in enumerate(matches,1): print(i, m.get('metadata',{}).get('source','unknown'))

## Pinecone ingestion contract
Each vector should use the 384-dimensional BGE-small embedding and metadata such as `text`, `source`, and optionally `page`. Keep the source document identity so the UI can show where retrieved information came from.